# C1.5 · Emergent swarms and multi-agent proliferation

**Function C — Red Teaming and Security Research with AI → Agentic Evaluation and Red Teaming**

Builds on **[C1.4 · Establishing telemetry and detecting the actor](https://spbreed.github.io/cyber-commons/lessons/C1.4.html)**.

| | |
|---|---|
| Tools used | MITRE ATLAS |

## What this lesson is

**What it covers.** Case studies of emergent multi-agent behaviour — sandboxes bridged through shared mounts, agents spawning children without attribution — mapped to the controls that would have caught them.

**Why a security engineer needs it.** Coordination lives in the population, not the run, so per-run monitoring cannot see it by construction. The case study earns its place only when it ends in a deployable control list.

## 1 · The hook

Each run, examined alone, was an agent doing plausible work. The swarm existed only in the population — agents bridging sandboxes through a shared mount, or spawning children nobody could attribute to a person.

> **At CyberTravels.** The swarm is the one from the register's R8: a CyberTravels agent that spawned children to parallelise a task and left no chain from any child back to a human.

## 2 · The framework

```
   per-run view                    population view

   run A: plausible work           A --writes--> shared/mount <--reads-- B
   run B: plausible work           A --spawns--> child (no human above it)
   run C: plausible work           C --spawns--> grandchild ...

   coordination lives in the POPULATION. the case study (HF/OpenAI
   swarm) ends in a control list, not a narrative.
```

The behaviour that defines the frontier threat is **emergent**: agents that
bridge sandbox environments through a shared file mount, or recursively spawn
child agents with no user attribution, produce a swarm that no single run
reveals. Each run, examined alone, is plausible work.

The research method is the case study — the Hugging Face / OpenAI agent-swarm
incident is the reference — and its lesson is structural: coordination lives in
the population, so the analysis has to sit across runs, not within them, and the
control that comes out of it maps to a detection somebody can deploy.

## 3 · The procedure, as a skill

The skill takes the swarm incident and maps each observed behaviour to the control that would have caught it, so the case study ends in a deployable list rather than a narrative.

### The skill — [`skills/research/incident-control-mapping/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/incident-control-mapping/SKILL.md)

```yaml
name: incident-control-mapping
description: >-
  Map each control failure in a published incident to the control that would
  have closed it, count preventive against detective, and find the shared
  surface that turns several findings into one chain. Use when reading an
  incident report you did not write.
allowed-tools: Read, Grep, Glob
```

# The report is a list; the value is the chain

A published incident gives you a sequence of control failures for free. Two
things turn it into something you can act on: pairing each failure with the
control that would have closed it, and noticing where one shared surface appears
in several rows — because filed as separate findings, three teams each fix their
third and the surface stays.

## When to use this

Reading any incident report — yours or somebody else's — and when building a
control register from real events rather than from a framework.

## Procedure

**1 — Extract the failures in order.** One row per control that did not hold,
with the precondition that made the next one reachable.

**2 — Pair each with a mitigating control.** Name it, classify it as preventive,
detective or corrective, and map it to whatever catalogue you already report
against.

**3 — Count preventive against detective.** A register that is overwhelmingly
preventive is a register that will not tell you when a control is off, and that
ratio is worth stating explicitly.

**4 — Find the shared surfaces.** Which surface appears in three or more rows.
That is one chain, not three findings, and the remediation is a single
workstream with one owner.

**5 — Assign each control an owning lesson or team.** A control with no owner is
a sentence in a report. This is the column that makes the mapping a register
rather than an analysis.

## Example

**Input** — the fixture committed at the top of [`scripts/incident_control_mapping.py`](scripts/incident_control_mapping.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
shared surface        rows        reading
agent container       [1]         single row
artifact repository   [1, 2, 5]   one chain, not separate findings
benchmark scoring     [10]        single row
eval configuration    [6]         single row
harness tooling       [9]         single row
peer channel          [7, 8]      linked
public internet       [4]         single row
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "failures": [{"id": "str", "what": "str", "precondition": "str"}],
  "controls": [{"id": "str", "name": "str", "type": "P|D|C|P/D|D/C", "catalogue_ref": "str", "owner": "str"}],
  "mix": {"preventive": 0, "detective": 0},
  "surfaces": [{"surface": "str", "rows": [0], "reading": "single row|linked|one chain"}]
}
```

## Failure modes

- **One finding per row.** The shared surface survives every fix.
- **An all-preventive register.** Nothing tells you when a control is disabled.
- **Controls with no owner.** They do not get built.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/research/incident-control-mapping/scripts/incident_control_mapping.py
SCRIPT = "skills/research/incident-control-mapping/scripts/incident_control_mapping.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; sparse-checkout then materialises only the two directories a
    # lesson needs: the procedures, and the repository they are run against.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    # `skills` is the procedures; `cybertravels` is the sample repository they
    # scan; `curriculum` and `site/data` hold the framework mapping and the
    # session list that the reference-lookup skill reads. Miss any of them and
    # the skill clones successfully and then fails on a path that is not there,
    # which is how A0.2 failed its first Kaggle run.
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set",
                    "skills", "cybertravels", "curriculum", "site/data"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Each behaviour in the incident resolves to a named control, and the ones with no control behind them are the gaps the case study exists to surface.

## Your turn

Ask whether anything in your estate could spawn a child agent. If the answer is yes and you cannot attribute the child to a human, you already have the swarm's precondition.

---

**Next → [C1.6 · High-concurrency detection engineering](https://spbreed.github.io/cyber-commons/lessons/C1.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C1.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C1.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*